In [1]:
from pathlib import Path
import sys, os, subprocess, zipfile, json

assert (3, 10) <= sys.version_info[:2] <= (3, 12), 'Dùng Python 3.10–3.12 cho bộ thư viện đã pin.'
BUNDLE_PATH = ''  # Kaggle: /kaggle/input/<private-input>/unifashion_week7_colab.zip
WORK = Path('/kaggle/working/unifashion_week7' if Path('/kaggle/working').exists() else '/content/unifashion_week7')
if not BUNDLE_PATH:
    from google.colab import files
    uploaded = files.upload()
    matches = [name for name in uploaded if name.endswith('.zip')]
    assert len(matches) == 1, 'Upload đúng một file unifashion_week7_colab.zip'
    BUNDLE_PATH = matches[0]
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    for member in archive.infolist():
        destination = (WORK / member.filename).resolve()
        assert destination.is_relative_to(WORK.resolve()), 'Unsafe ZIP path'
    archive.extractall(WORK)
os.chdir(WORK)
print('Working directory:', WORK)
subprocess.run(['nvidia-smi'], check=True)


Saving unifashion_week7_colab.zip to unifashion_week7_colab.zip
Working directory: /content/unifashion_week7


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [11]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-unifashion.txt'], check=True)
subprocess.run([sys.executable, '-c', "import torch; print('PyTorch:',torch.__version__); assert torch.cuda.is_available(), 'Enable GPU runtime'; p=torch.cuda.get_device_properties(0); print(p.name, 'VRAM GiB:',p.total_memory/2**30)"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch; print('PyTorch:',torch.__version__); assert torch.cuda.is_available(), 'Enable GPU runtime'; p=torch.cuda.get_device_properties(0); print(p.name, 'VRAM GiB:',p.total_memory/2**30)"], returncode=0)

In [2]:
def run_live(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise RuntimeError(f'Command failed with exit code {process.returncode}')

run_live([sys.executable, '-u', 'scripts/download_unifashion.py', '--with-vision'])


Manifest: /content/unifashion_week7/checkpoints/unifashion/download_manifest.json


In [6]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "-r", "requirements-unifashion.txt"
], check=True)

subprocess.run([
    sys.executable, "-c",
    "import transformers; "
    "print(transformers.__version__, transformers.__file__); "
    "assert transformers.__version__ == '4.36.2'; "
    "from transformers.modeling_utils import apply_chunking_to_forward"
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-c', "import transformers; print(transformers.__version__, transformers.__file__); assert transformers.__version__ == '4.36.2'; from transformers.modeling_utils import apply_chunking_to_forward"], returncode=0)

In [12]:
run_live([sys.executable, '-m', 'pytest', 'tests/test_unifashion_sanity.py', '-q'])
config = json.loads(Path('configs/unifashion_sanity.json').read_text())
audit = json.loads(Path('data/data_audit.json').read_text())
print(json.dumps(config, indent=2))
print('Train:', audit['splits']['train']['selected_rows'], 'Val:', audit['splits']['val']['selected_rows'])
print('Image overlap:', audit['selected_train_val_image_overlap'])
assert audit['selected_train_val_image_overlap'] == 0

....                                                                     [100%]
=============================== warnings summary ===============================
tests/test_unifashion_sanity.py::test_official_forward_backward_and_reload_with_tiny_fixture
  /usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
    _torch_pytree._register_pytree_node(

tests/test_unifashion_sanity.py::test_official_forward_backward_and_reload_with_tiny_fixture
tests/test_unifashion_sanity.py::test_official_forward_backward_and_reload_with_tiny_fixture
  /usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
    _torch_pytree._register_pytree_node(

-- Docs: https://docs.pytest.org/en/stable/how-t

In [9]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select the updated unifashion_runtime.py
assert set(uploaded) == {"unifashion_runtime.py"}

(Path(WORK) / "src/unifashion_runtime.py").write_bytes(
    uploaded["unifashion_runtime.py"]
)
print("Adapter updated. Rerun cell 5.")

Saving unifashion_runtime.py to unifashion_runtime.py
Adapter updated. Rerun cell 5.


In [14]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys, os, subprocess

# Define WORK and run_live if they are not already in scope from previous cells
# Assuming WORK is already defined from week7-02 and run_live from week7-06

# 1. Load the existing config
config_path = Path('configs/unifashion_sanity.json')
config_data = json.loads(config_path.read_text())

# 2. Modify the precision to fp32 for numerical stability
config_data['precision'] = 'fp32'

# 3. Save the modified config back
config_path.write_text(json.dumps(config_data, indent=2))

RUN_DIR = WORK / 'reports' / ('run_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
run_live([sys.executable, '-u', 'scripts/train_unifashion_sanity.py',
          '--data-dir', str(WORK / 'data'), '--images-dir', str(WORK / 'images'),
          '--output', str(RUN_DIR)])
status = json.loads((RUN_DIR / 'status.json').read_text())
print(json.dumps(status, indent=2))
assert status['status'] == 'sanity_passed'
assert status['epoch_completed'] and status['reload_verified']

Traceback (most recent call last):
  File "/content/unifashion_week7/scripts/train_unifashion_sanity.py", line 234, in main
    execute(args, output)
  File "/content/unifashion_week7/scripts/train_unifashion_sanity.py", line 85, in execute
    raise ValueError("This tested runner uses fp16 CUDA autocast")
ValueError: This tested runner uses fp16 CUDA autocast
Traceback (most recent call last):
  File "/content/unifashion_week7/scripts/train_unifashion_sanity.py", line 242, in <module>
    main()
  File "/content/unifashion_week7/scripts/train_unifashion_sanity.py", line 234, in main
    execute(args, output)
  File "/content/unifashion_week7/scripts/train_unifashion_sanity.py", line 85, in execute
    raise ValueError("This tested runner uses fp16 CUDA autocast")
ValueError: This tested runner uses fp16 CUDA autocast


RuntimeError: Command failed with exit code 1

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
metrics = pd.read_csv(RUN_DIR / 'train_metrics.csv')
ax = metrics.plot(x='step', y=['loss', 'loss_itc', 'loss_itm', 'loss_ttc'], marker='o', figsize=(9, 4))
ax.set_title('UniFashion: 1 epoch sanity check, 24 training triplets')
ax.set_ylabel('Loss'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / 'loss_curve.png', dpi=160)
plt.show()
print(metrics.to_string(index=False))


In [ ]:
import shutil
EVIDENCE = WORK.parent / (RUN_DIR.name + '_evidence.zip')
with zipfile.ZipFile(EVIDENCE, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in RUN_DIR.rglob('*'):
        if path.is_file(): archive.write(path, 'run/' + str(path.relative_to(RUN_DIR)))
    for path in (WORK / 'data').glob('*'):
        if path.is_file(): archive.write(path, 'data/' + path.name)
    for folder in ('scripts', 'src', 'configs', 'tests'):
        for path in (WORK / folder).rglob('*'):
            if path.is_file() and '__pycache__' not in path.parts:
                archive.write(path, str(path.relative_to(WORK)))
    archive.write(WORK / 'requirements-unifashion.txt', 'requirements-unifashion.txt')
print('Evidence:', EVIDENCE)
checkpoint = Path(status['checkpoint'])
print('Checkpoint:', checkpoint, f'({checkpoint.stat().st_size / 2**30:.2f} GiB)')
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(str(EVIDENCE))


In [ ]:
# Colab: chạy cell này để lưu checkpoint lớn vào Drive của bạn.
if not Path('/kaggle/working').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    destination = Path('/content/drive/MyDrive/Thesis/UniFashion') / RUN_DIR.name
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(checkpoint, destination / checkpoint.name)
    shutil.copy2(EVIDENCE, destination / EVIDENCE.name)
    print('Saved:', destination)
